# GitHub Trending Dashboard

In [1]:
!pip install requests beautifulsoup4 lxml pandas dash plotly -q

In [2]:
import re
import requests
import pandas as pd
import plotly.express as px
from bs4 import BeautifulSoup
from dash import Dash, dcc, html, Input, Output

## 2. Scrape the data

In [3]:
def to_int(text):
    digits = re.sub(r"[^\d]", "", text or "")
    return int(digits) if digits else 0


def scrape(period):
    url = "https://github.com/trending"
    page = requests.get(url, params={"since": period}, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    soup = BeautifulSoup(page.text, "lxml")

    rows = []
    for rank, repo in enumerate(soup.select("article.Box-row"), start=1):
        name = re.sub(r"\s+", "", repo.select_one("h2 a").get_text())
        lang = repo.select_one("[itemprop=programmingLanguage]")
        stars = repo.select_one("a[href$='/stargazers']")
        forks = repo.select_one("a[href$='/forks']")
        today = repo.select_one("span.float-sm-right")
        rows.append({
            "period": period,
            "rank": rank,
            "repo": name,
            "language": lang.get_text(strip=True) if lang else "Unknown",
            "stars": to_int(stars.get_text()) if stars else 0,
            "forks": to_int(forks.get_text()) if forks else 0,
            "stars_gained": to_int(today.get_text()) if today else 0,
        })
    return rows


data = []
for p in ["daily", "weekly", "monthly"]:
    data += scrape(p)

df = pd.DataFrame(data)
df.head()

,period,rank,repo,language,stars,forks,stars_gained
0,daily,1,morluto/rea,TypeScript,68502,14425,25784
1,daily,2,boykopovar/AnyPS5,C++,26073,2181,5831
2,daily,3,storytold/artcraft,Rust,13918,2178,3217
3,daily,4,cathrynlavery/diagram-design,HTML,48749,3077,1189
4,daily,5,mksglu/context-mode,TypeScript,26224,1888,178


## 3. Save to CSV

In [4]:
df.to_csv("trending_repos.csv", index=False)
print(len(df), "rows saved to trending_repos.csv")

49 rows saved to trending_repos.csv


In [8]:
app = Dash(__name__)

app.layout = html.Div(style={"padding": "20px", "fontFamily": "Arial"}, children=[
    html.H1("GitHub Trending Repositories"),

    # Filter
    dcc.Dropdown(id="period", options=["daily", "weekly", "monthly"], value="daily",
                 clearable=False, style={"width": "250px"}),

    # KPI cards
    html.Div(id="cards", style={"display": "flex", "gap": "15px", "margin": "20px 0"}),

    # Charts
    dcc.Graph(id="top10"),
    dcc.Graph(id="languages"),
])


def card(title, value):
    return html.Div([html.Div(title, style={"color": "gray"}), html.H2(value)],
                    style={"border": "1px solid #ddd", "borderRadius": "8px",
                           "padding": "10px 20px", "flex": "1"})


@app.callback(Output("cards", "children"), Output("top10", "figure"), Output("languages", "figure"),
              Input("period", "value"))
def update(period):
    d = df[df["period"] == period]

    cards = [
        card("Repositories", len(d)),
        card("Total stars", f"{d['stars'].sum():,}"),
        card("Total forks", f"{d['forks'].sum():,}"),
        card("Top language", d["language"].value_counts().idxmax()),
    ]

    top10 = d.nlargest(10, "stars").sort_values("stars")
    fig1 = px.bar(top10, x="stars", y="repo", orientation="h", title="Top 10 trending repositories")

    langs = d["language"].value_counts().reset_index()
    langs.columns = ["language", "count"]
    fig2 = px.pie(langs, names="language", values="count", title="Programming language distribution")

    return cards, fig1, fig2


app.run(debug=False , jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/
